In [3]:
!pip install pandas numpy scikit-learn gradio joblib matplotlib seaborn

# IMPORT LIBARIES

In [4]:
import pandas as pd
import numpy as np
import gradio as gr
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# LOAD DATSET

In [5]:
df = pd.read_csv("public_transport_demand_dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (100000, 22)


,record_id,timestamp,date,hour,day_of_week,day_name,is_weekend,is_holiday,is_special_event,is_peak_hour,...,stop_name,distance_to_cbd_km,weather_condition,temperature_c,precipitation_mm,humidity_pct,fare_inr,scheduled_capacity,passenger_count,occupancy_rate_pct
0,1,2024-01-01 00:00:00,2024-01-01,0,0,Monday,0,0,0,0,...,Stop_143,4.28,Clear,16.6,0.0,67,14.0,60,25,41.7
1,2,2024-01-01 00:00:00,2024-01-01,0,0,Monday,0,1,0,0,...,Stop_161,6.02,Clear,15.5,0.0,79,17.0,60,11,18.3
2,3,2024-01-01 00:00:00,2024-01-01,0,0,Monday,0,0,0,0,...,Stop_98,4.90,Clear,28.6,0.0,67,15.0,60,23,38.3
3,4,2024-01-01 00:45:00,2024-01-01,0,0,Monday,0,0,0,0,...,Stop_99,8.60,Cloudy,18.7,0.0,37,15.0,60,19,31.7
4,5,2024-01-01 00:45:00,2024-01-01,0,0,Monday,0,0,0,0,...,Stop_9,6.81,Rain,27.4,2.8,51,16.0,60,18,30.0


# MISSING VALUES

In [6]:
print(df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

['record_id', 'timestamp', 'date', 'hour', 'day_of_week', 'day_name', 'is_weekend', 'is_holiday', 'is_special_event', 'is_peak_hour', 'route_id', 'transport_mode', 'stop_name', 'distance_to_cbd_km', 'weather_condition', 'temperature_c', 'precipitation_mm', 'humidity_pct', 'fare_inr', 'scheduled_capacity', 'passenger_count', 'occupancy_rate_pct']

Missing values:
record_id             0
timestamp             0
date                  0
hour                  0
day_of_week           0
day_name              0
is_weekend            0
is_holiday            0
is_special_event      0
is_peak_hour          0
route_id              0
transport_mode        0
stop_name             0
distance_to_cbd_km    0
weather_condition     0
temperature_c         0
precipitation_mm      0
humidity_pct          0
fare_inr              0
scheduled_capacity    0
passenger_count       0
occupancy_rate_pct    0
dtype: int64


# DROPNA

In [7]:
df = df.dropna().reset_index(drop=True)

print("Dataset shape after cleaning:", df.shape)

Dataset shape after cleaning: (100000, 22)


In [9]:
print(df.columns.tolist())

['record_id', 'timestamp', 'date', 'hour', 'day_of_week', 'day_name', 'is_weekend', 'is_holiday', 'is_special_event', 'is_peak_hour', 'route_id', 'transport_mode', 'stop_name', 'distance_to_cbd_km', 'weather_condition', 'temperature_c', 'precipitation_mm', 'humidity_pct', 'fare_inr', 'scheduled_capacity', 'passenger_count', 'occupancy_rate_pct']


In [10]:
features = [
    "date",
    "hour",
    "day_of_week",
    "day_name",
    "is_weekend",
    "is_holiday",
    "is_special_event",
    "is_peak_hour",
    "route_id",
    "transport_mode",
    "stop_name",
    "distance_to_cbd_km",
    "weather_condition",
    "temperature_c",
    "precipitation_mm",
    "humidity_pct",
    "fare_inr",
    "scheduled_capacity"
]

target = "passenger_count"

X = df[features].copy()
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (100000, 18)
y shape: (100000,)


In [11]:
X["date"] = pd.to_datetime(X["date"])

X["year"] = X["date"].dt.year
X["month"] = X["date"].dt.month
X["day"] = X["date"].dt.day

X = X.drop(columns=["date"])

print(X.head())

   hour  day_of_week day_name  is_weekend  is_holiday  is_special_event  \
0     0            0   Monday           0           0                 0   
1     0            0   Monday           0           1                 0   
2     0            0   Monday           0           0                 0   
3     0            0   Monday           0           0                 0   
4     0            0   Monday           0           0                 0   

   is_peak_hour route_id transport_mode stop_name  distance_to_cbd_km  \
0             0     R019            Bus  Stop_143                4.28   
1             0     R022            Bus  Stop_161                6.02   
2             0     R005            Bus   Stop_98                4.90   
3             0     R001            Bus   Stop_99                8.60   
4             0     R034            Bus    Stop_9                6.81   

  weather_condition  temperature_c  precipitation_mm  humidity_pct  fare_inr  \
0             Clear           

In [12]:
categorical_features = [
    "day_name",
    "route_id",
    "transport_mode",
    "stop_name",
    "weather_condition"
]

numerical_features = [
    "hour",
    "day_of_week",
    "is_weekend",
    "is_holiday",
    "is_special_event",
    "is_peak_hour",
    "distance_to_cbd_km",
    "temperature_c",
    "precipitation_mm",
    "humidity_pct",
    "fare_inr",
    "scheduled_capacity",
    "year",
    "month",
    "day"
]

print("Categorical:", categorical_features)
print("Numerical:", numerical_features)

Categorical: ['day_name', 'route_id', 'transport_mode', 'stop_name', 'weather_condition']
Numerical: ['hour', 'day_of_week', 'is_weekend', 'is_holiday', 'is_special_event', 'is_peak_hour', 'distance_to_cbd_km', 'temperature_c', 'precipitation_mm', 'humidity_pct', 'fare_inr', 'scheduled_capacity', 'year', 'month', 'day']


In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

print("Pipeline created successfully!")

Pipeline created successfully!


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (80000, 20)
Testing: (20000, 20)


In [15]:
pipeline.fit(X_train, y_train)

print("✅ Model training completed!")

✅ Model training completed!


In [16]:
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("===== MODEL PERFORMANCE =====")
print("MAE  :", round(mae, 2))
print("RMSE :", round(rmse, 2))
print("R²   :", round(r2, 3))

===== MODEL PERFORMANCE =====
MAE  : 6.55
RMSE : 11.79
R²   : 0.968


In [17]:
joblib.dump(pipeline, "public_transport_demand_model.pkl")

print("✅ Model saved!")

✅ Model saved!


In [18]:
days = sorted(df["day_name"].unique().tolist())
routes = sorted(df["route_id"].unique().tolist())
transport_modes = sorted(df["transport_mode"].unique().tolist())
stops = sorted(df["stop_name"].unique().tolist())
weather = sorted(df["weather_condition"].unique().tolist())

print("Days:", days)
print("Routes:", routes)
print("Transport modes:", transport_modes)
print("Stops:", stops)
print("Weather:", weather)

Days: ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']
Routes: ['R001', 'R002', 'R003', 'R004', 'R005', 'R006', 'R007', 'R008', 'R009', 'R010', 'R011', 'R012', 'R013', 'R014', 'R015', 'R016', 'R017', 'R018', 'R019', 'R020', 'R021', 'R022', 'R023', 'R024', 'R025', 'R026', 'R027', 'R028', 'R029', 'R030', 'R031', 'R032', 'R033', 'R034', 'R035', 'R036', 'R037', 'R038', 'R039', 'R040', 'R041', 'R042', 'R043', 'R044', 'R045', 'R046', 'R047', 'R048', 'R049', 'R050']
Transport modes: ['Bus', 'Metro', 'Suburban Train', 'Tram']
Stops: ['Stop_1', 'Stop_10', 'Stop_100', 'Stop_101', 'Stop_102', 'Stop_103', 'Stop_104', 'Stop_105', 'Stop_106', 'Stop_107', 'Stop_108', 'Stop_109', 'Stop_11', 'Stop_110', 'Stop_111', 'Stop_112', 'Stop_113', 'Stop_114', 'Stop_115', 'Stop_116', 'Stop_117', 'Stop_118', 'Stop_119', 'Stop_12', 'Stop_120', 'Stop_121', 'Stop_122', 'Stop_123', 'Stop_124', 'Stop_125', 'Stop_126', 'Stop_127', 'Stop_128', 'Stop_129', 'Stop_13', 'Stop_130', 'Stop_131', 

In [19]:
def predict_demand(
    date,
    hour,
    day_of_week,
    day_name,
    weekend,
    holiday,
    special_event,
    peak_hour,
    route_id,
    transport_mode,
    stop_name,
    distance_cbd,
    weather_condition,
    temperature,
    precipitation,
    humidity,
    fare,
    scheduled_capacity
):

    data = pd.DataFrame([{
        "date": pd.to_datetime(date),
        "hour": hour,
        "day_of_week": day_of_week,
        "day_name": day_name,
        "is_weekend": weekend,
        "is_holiday": holiday,
        "is_special_event": special_event,
        "is_peak_hour": peak_hour,
        "route_id": route_id,
        "transport_mode": transport_mode,
        "stop_name": stop_name,
        "distance_to_cbd_km": distance_cbd,
        "weather_condition": weather_condition,
        "temperature_c": temperature,
        "precipitation_mm": precipitation,
        "humidity_pct": humidity,
        "fare_inr": fare,
        "scheduled_capacity": scheduled_capacity
    }])

    data["date"] = pd.to_datetime(data["date"])

    data["year"] = data["date"].dt.year
    data["month"] = data["date"].dt.month
    data["day"] = data["date"].dt.day

    data = data.drop(columns=["date"])

    prediction = pipeline.predict(data)[0]

    prediction = max(0, round(prediction))

    if prediction < 50:
        level = "🟢 Low Demand"
        recommendation = "Normal service is sufficient."

    elif prediction < 100:
        level = "🟡 Medium Demand"
        recommendation = "Consider increasing service frequency."

    else:
        level = "🔴 High Demand"
        recommendation = "Increase vehicles or service frequency."

    return (
        f"{prediction} passengers",
        level,
        recommendation
    )

In [20]:
demo = gr.Interface(
    fn=predict_demand,

    inputs=[
        gr.Textbox(
            value="2024-01-01",
            label="Date"
        ),

        gr.Slider(
            0, 23,
            value=8,
            step=1,
            label="Hour"
        ),

        gr.Slider(
            0, 6,
            value=0,
            step=1,
            label="Day of Week"
        ),

        gr.Dropdown(
            choices=days,
            value=days[0],
            label="Day Name"
        ),

        gr.Radio(
            [0, 1],
            value=0,
            label="Weekend?"
        ),

        gr.Radio(
            [0, 1],
            value=0,
            label="Holiday?"
        ),

        gr.Radio(
            [0, 1],
            value=0,
            label="Special Event?"
        ),

        gr.Radio(
            [0, 1],
            value=1,
            label="Peak Hour?"
        ),

        gr.Dropdown(
            choices=routes,
            value=routes[0],
            label="Route ID"
        ),

        gr.Dropdown(
            choices=transport_modes,
            value=transport_modes[0],
            label="Transport Mode"
        ),

        gr.Dropdown(
            choices=stops,
            value=stops[0],
            label="Stop Name"
        ),

        gr.Number(
            value=5,
            label="Distance to CBD (km)"
        ),

        gr.Dropdown(
            choices=weather,
            value=weather[0],
            label="Weather Condition"
        ),

        gr.Number(
            value=28,
            label="Temperature (°C)"
        ),

        gr.Number(
            value=0,
            label="Precipitation (mm)"
        ),

        gr.Number(
            value=70,
            label="Humidity (%)"
        ),

        gr.Number(
            value=20,
            label="Fare (₹)"
        ),

        gr.Number(
            value=100,
            label="Scheduled Capacity"
        )
    ],

    outputs=[
        gr.Textbox(label="Predicted Demand"),
        gr.Textbox(label="Demand Level"),
        gr.Textbox(label="Recommendation")
    ],

    title="🚌 Public Transport Demand Prediction",
    description="ML-Based Passenger Demand Prediction System"
)

In [1]:
demo.launch()

NameError: name 'demo' is not defined